In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
CONCATENATION DES PANELS SOLDE 2015-2020 ET 2021-2025
"""

import boto3
import io
import pandas as pd
from io import BytesIO

# =============================================================================
# CONNEXION S3
# =============================================================================
print("="*80)
print("🔗 CONCATENATION DES PANELS SOLDE")
print("="*80 + "\n")

print("⚙️  Connexion à MinIO/S3...")
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
    aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
    verify=False
)

bucket_name = "admindataanstat"

print("✅ Connexion établie\n")

# =============================================================================
# CHARGEMENT DES DEUX BASES
# =============================================================================

print("📂 Chargement des fichiers Parquet...\n")

# Fichier 1 : 2015-2020
file1_key = "Solde/panel_solde_1_2_2015_2020.parquet"
print(f"🔄 Lecture: {file1_key}")

try:
    obj1 = s3_client.get_object(Bucket=bucket_name, Key=file1_key)
    df1 = pd.read_parquet(BytesIO(obj1["Body"].read()))
    print(f"✅ Panel 2015-2020: {len(df1):,} lignes, {df1.shape[1]} colonnes")
    print(f"   Colonnes: {list(df1.columns)[:5]}...")
    
    # Identifier la période
    if 'PERIODE' in df1.columns:
        periodes1 = df1['PERIODE'].unique()
        print(f"   Périodes: {len(periodes1)} mois ({min(periodes1)} → {max(periodes1)})")
    
except Exception as e:
    print(f"❌ Erreur lecture fichier 1: {e}")
    exit(1)

# Fichier 2 : 2021-2025
file2_key = "Solde/panel_solde_1_2_2021_2025"
print(f"\n🔄 Lecture: {file2_key}")

try:
    obj2 = s3_client.get_object(Bucket=bucket_name, Key=file2_key)
    df2 = pd.read_parquet(BytesIO(obj2["Body"].read()))
    print(f"✅ Panel 2021-2025: {len(df2):,} lignes, {df2.shape[1]} colonnes")
    print(f"   Colonnes: {list(df2.columns)[:5]}...")
    
    # Identifier la période
    if 'PERIODE' in df2.columns:
        periodes2 = df2['PERIODE'].unique()
        print(f"   Périodes: {len(periodes2)} mois ({min(periodes2)} → {max(periodes2)})")
    
except Exception as e:
    print(f"❌ Erreur lecture fichier 2: {e}")
    exit(1)


In [ ]:

# =============================================================================
# ANALYSE DES COLONNES
# =============================================================================

print(f"\n{'='*80}")
print("📊 ANALYSE DES COLONNES")
print("="*80)

cols1 = set(df1.columns)
cols2 = set(df2.columns)

cols_communes = cols1 & cols2
cols_uniques_df1 = cols1 - cols2
cols_uniques_df2 = cols2 - cols1

print(f"   • Colonnes communes       : {len(cols_communes)}")
print(f"   • Colonnes uniques 2015-2020 : {len(cols_uniques_df1)}")
print(f"   • Colonnes uniques 2021-2025 : {len(cols_uniques_df2)}")

if cols_uniques_df1:
    print(f"\n   🔍 Colonnes uniques dans 2015-2020:")
    for col in sorted(list(cols_uniques_df1)[:10]):
        print(f"      • {col}")
    if len(cols_uniques_df1) > 10:
        print(f"      ... et {len(cols_uniques_df1) - 10} autres")

if cols_uniques_df2:
    print(f"\n   🔍 Colonnes uniques dans 2021-2025:")
    for col in sorted(list(cols_uniques_df2)[:10]):
        print(f"      • {col}")
    if len(cols_uniques_df2) > 10:
        print(f"      ... et {len(cols_uniques_df2) - 10} autres")



In [ ]:
# =============================================================================
# HARMONISATION DES COLONNES
# =============================================================================

print(f"\n{'='*80}")
print("🔧 HARMONISATION DES COLONNES")
print("="*80)

# Obtenir toutes les colonnes uniques
all_columns = sorted(cols1 | cols2)
print(f"   • Total colonnes uniques: {len(all_columns)}")

# Réindexer les deux DataFrames avec toutes les colonnes
print(f"\n   🔄 Réindexation des DataFrames...")
df1_harmonise = df1.reindex(columns=all_columns)
df2_harmonise = df2.reindex(columns=all_columns)

print(f"   ✅ DataFrame 1: {df1_harmonise.shape}")
print(f"   ✅ DataFrame 2: {df2_harmonise.shape}")



In [ ]:
# =============================================================================
# CONCATENATION
# =============================================================================

print(f"\n{'='*80}")
print("🔗 CONCATENATION")
print("="*80)

print(f"   🔄 Concaténation en cours...")
panel_complet = pd.concat([df1_harmonise, df2_harmonise], ignore_index=True)

print(f"   ✅ Panel complet: {len(panel_complet):,} lignes, {panel_complet.shape[1]} colonnes")

# Vérifier la période complète
if 'PERIODE' in panel_complet.columns:
    periodes_completes = panel_complet['PERIODE'].unique()
    print(f"   📅 Périodes totales: {len(periodes_completes)} mois")
    print(f"      • Première période: {min(periodes_completes)}")
    print(f"      • Dernière période: {max(periodes_completes)}")



In [ ]:
# =============================================================================
# STATISTIQUES
# =============================================================================

print(f"\n{'='*80}")
print("📊 STATISTIQUES DU PANEL COMPLET")
print("="*80)

print(f"\n   📈 Taille des données:")
print(f"      • Lignes: {len(panel_complet):,}")
print(f"      • Colonnes: {panel_complet.shape[1]}")
print(f"      • Mémoire: {panel_complet.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

if 'PERIODE' in panel_complet.columns:
    print(f"\n   📅 Distribution par année:")
    panel_complet['ANNEE'] = panel_complet['PERIODE'].str[-4:]
    annees_stats = panel_complet['ANNEE'].value_counts().sort_index()
    for annee, count in annees_stats.items():
        print(f"      • {annee}: {count:,} lignes")
    panel_complet = panel_complet.drop(columns=['ANNEE'])

# Vérifier les doublons potentiels
if 'MATRICULE||CODE_ORGANISME' in panel_complet.columns and 'PERIODE' in panel_complet.columns:
    print(f"\n   🔍 Vérification des doublons:")
    nb_doublons = panel_complet.duplicated(subset=['MATRICULE||CODE_ORGANISME', 'PERIODE']).sum()
    print(f"      • Doublons (MATRICULE||CODE_ORGANISME + PERIODE): {nb_doublons:,}")
    
    if nb_doublons > 0:
        print(f"      ⚠️  Suppression des doublons...")
        panel_complet = panel_complet.drop_duplicates(
            subset=['MATRICULE||CODE_ORGANISME', 'PERIODE'],
            keep='first'
        )
        print(f"      ✅ Panel après dédoublonnage: {len(panel_complet):,} lignes")



In [ ]:
# =============================================================================
# EXPORT
# =============================================================================

print(f"\n{'='*80}")
print("💾 EXPORT VERS S3")
print("="*80)

output_key = "Solde/out/panel_solde_complet_2015_2025.parquet"

print(f"\n   🔄 Export vers: {output_key}")

try:
    parquet_buffer = io.BytesIO()
    panel_complet.to_parquet(parquet_buffer, engine="pyarrow", index=False)
    parquet_buffer.seek(0)
    
    s3_client.put_object(
        Bucket=bucket_name,
        Key=output_key,
        Body=parquet_buffer.getvalue()
    )
    
    size_mb = len(parquet_buffer.getvalue()) / (1024**2)
    
    print(f"   ✅ Fichier exporté avec succès!")
    print(f"      • Fichier: {output_key}")
    print(f"      • Taille: {size_mb:.2f} MB")
    print(f"      • Lignes: {len(panel_complet):,}")
    print(f"      • Colonnes: {panel_complet.shape[1]}")
    
except Exception as e:
    print(f"   ❌ Erreur lors de l'export: {e}")
    exit(1)



In [ ]:
# =============================================================================
# RÉSUMÉ FINAL
# =============================================================================

print(f"\n{'='*80}")
print("✅ CONCATENATION TERMINÉE AVEC SUCCÈS")
print("="*80)

print(f"\n📦 Fichiers:")
print(f"   • Input 1: {file1_key}")
print(f"   • Input 2: {file2_key}")
print(f"   • Output : {output_key}")

print(f"\n📊 Résumé:")
print(f"   • 2015-2020: {len(df1):,} lignes")
print(f"   • 2021-2025: {len(df2):,} lignes")
print(f"   • Total    : {len(panel_complet):,} lignes")
print(f"   • Colonnes : {panel_complet.shape[1]}")

print(f"\n🎯 Panel complet disponible sur S3:")
print(f"   s3://{bucket_name}/{output_key}")